# U7-L4：多模态流程衔接

**课程目标**

1. 能为感知、播报和行为片段写出交接条件，说明播报之后为何需要新观测。
2. 能分别运行导览和足球的局部参考片段，按所用接口管理任务句柄或自动踢球控制。
3. 能记录程序返回原因、SDK状态和现场表现，未完成时停止后续操作，并核查重复运行条件。

## 一、任务：一句播报之后还能按原计划行动吗

定点导览发现来宾后准备介绍并致意；足球场景发现球后准备说明并试踢。两者都需要视觉、语音和行为衔接，但播报结束时目标可能已经移动或离开。

本课比较两个局部片段：`确认目标 → 播报 → 重新确认 → 一次行为`。只运行这个交接过程，记录在哪里结束。导览暂不增加路线，足球不增加比赛策略。

| 交接点 | 导览 | 足球 |
| --- | --- | --- |
| 首次确认 | 单名来宾可见 | 足球在已核对的标定范围 |
| 播报后新观测 | 仍有来宾才致意 | 球坐标仍可用才进入管理器 |
| 行为结束 | 检查动作句柄实际状态及现场表现 | stop返回，并现场记录球的效果 |

### 1.1 多模态协作不要求所有步骤同时运行

L3展示了播报期间继续检测的方式。若片段只需在播报结束后重新确认，顺序程序就能满足需求；若要求期间持续观察，才增加工作线程和退出管理。本课先用顺序片段把交接条件核对清楚。多模态指信息与行为方式的协作，不等于必须多线程。

## 二、准备功能片段

⚠️ **实验安全**：致意和踢球均可能带动机身。两类片段分开运行，先清理上一类任务，再核对各自场地。踢球场地、标定与保护要求同L2；不在踢球管理器运行时另发速度、轨迹或动作指令。

📌 **前置**：L3线程与音频已结束；拟用动作已验证模式、时长和范围；踢球需有`ball_calibration.json`及足球步态下的教师验证。

**STEP 01：首次连接并保存会话记录。** `session`附件记录当前音频、动作及踢球管理器，重试前用它检查旧任务，不重跑连接格覆盖它。

In [ ]:
import json
from pathlib import Path
from boosteros.robots.booster import BoosterRobot
from boosteros.brain import Detection, Speech
from x_lab.flows import new_session, idle, observe_guest, observe_ball, say, greet, kick_window, cleanup
import time

robot = BoosterRobot()
speech = Speech(robot)
guest_detector = Detection(model="person")
ball_detector = Detection(model="soccer")
session = new_session(robot, speech)
calibration = None
if Path("ball_calibration.json").exists():
    calibration = json.loads(Path("ball_calibration.json").read_text(encoding="utf-8"))
for action in robot.list_actions():
    print(action.id, action.type, action.duration)

### 2.1 用返回原因决定交接

每个片段返回“完成”或具体停止原因。有限文本合成、播放、动作有任务句柄；自动踢球管理器的四个控制调用返回`None`。不能给所有接口套上相同的`wait()`。

附件`x_lab/flows.py`提供已学功能片段：`say()`合成与播放，`greet()`等待动作，`kick_window()`持续检测并更新球位。它们保存旧句柄、限制等待、必要时取消或停止，供本课和应用项目复用。可打开附件核对实际SDK调用，不把附件名称当作SDK接口。

`greet()`仅在`SUCCEEDED`时返回“完成”；超时取消会返回实际原因。某些已实测动作采用等待后取消时，应分别记录现场动作与取消终态，不把`CANCELLED`改写为成功；本次未获成功状态则结束片段，待教师审定该动作的具体交接约定。

**STEP 02：定义导览局部片段。** 来宾在通道外站定；本片段不驱动接近，不判断人物身份。首次及播报后的观察都检查本次取图与处理时长，0.8秒是待本机核对的练习参数；取得新源图像后才进入行为。

In [ ]:
GREET_ID = ""  # 从实际动作清单填写已验证的ID

def guide_segment():
    first = observe_guest(robot, guest_detector)
    if not first["found"]:
        return "首次未检出来宾"
    if time.monotonic()-first["received"] > 0.8:
        return "首次取图与处理过慢"
    result = say(session, "欢迎参观。请在指定区域观看机器人的致意动作。")
    if result != "完成":
        return "播报：" + result
    current = observe_guest(robot, guest_detector)
    if not current["found"]:
        return "播报后未检出来宾"
    if time.monotonic()-current["received"] > 0.8:
        return "播报后取图与处理过慢"
    if current["source_stamp"] == first["source_stamp"]:
        return "播报后尚无新图像"
    return greet(session, GREET_ID)

**STEP 03：定义足球局部片段。** 播报时管理器尚未开始；之后重新核对球位，管理器启动后仍需逐轮取新坐标。这里只设六秒观察窗口，不保证六秒内踢出或进球。

In [ ]:
def soccer_segment():
    if calibration is None or not calibration.get("verified", False):
        return "没有通过独立核对的标定"
    first = observe_ball(robot, ball_detector, calibration)
    if not first["found"]:
        return first["reason"]
    if time.monotonic()-first["received"] > 0.8:
        return "首次取图与处理过慢"
    result = say(session, "准备进行短时试踢，请留在场地外观看。")
    if result != "完成":
        return "播报：" + result
    current = observe_ball(robot, ball_detector, calibration)
    if not current["found"]:
        return "播报后：" + current["reason"]
    if time.monotonic()-current["received"] > 0.8:
        return "播报后取图与处理过慢"
    if current["source_stamp"] == first["source_stamp"]:
        return "播报后尚无新图像"
    reason, rows = kick_window(session, ball_detector, calibration, seconds=6.0, power=1.0)
    print("球坐标更新条数：", len(rows))
    return reason

## 三、一次只验证一类交接

**STEP 04：先选导览，再按条件安排足球。** 定义格不会运动；执行格会发起实际任务。按按钮流程先进入准备模式、等站稳，再进入本机实验要求的行走模式。动作ID和模式应在现场复核。

<font color="red">导览确认动作范围；足球另确认标定姿态、动态适用范围、球位与出球方向。旧任务未结束时不试演。</font>

In [ ]:
SCENE = "导览"  # 另一选项为“踢球”
APPROVED = False  # 每次实际条件确认后修改
if not APPROVED or robot.get_mode() != "walk":
    print("尚未满足本次运行条件")
elif not idle(session):
    print("旧任务尚未结束")
elif SCENE not in ["导览", "踢球"]:
    print("请填写导览或踢球")
elif input("本次可运行，输入“运行”：").strip() == "运行":
    try:
        if SCENE == "导览":
            reason = guide_segment()
        else:
            reason = soccer_segment()
        print("片段返回：", reason)
    finally:
        print("收尾待处理项：", cleanup(session))

**STEP 05：核查后再重试。** 返回原因表示程序条件，实际动作、来宾反应和球的去向另行记录。未确认结束时保留session，稍后重跑核查格；全部退出、人工恢复站位后才可重试。

In [ ]:
print("待处理项：", cleanup(session))
print("资源全部结束：", idle(session))
for key in ["audio_task", "motion_task"]:
    task = session[key]
    if task is not None:
        print(key, task.status, "done=", task.done())
print("踢球控制仍被本程序标为活动：", session["kick_active"])

本次结束先切准备模式并等站稳；需要坐下时按人工支撑、切阻尼、坐稳的既定流程处理，不能从行走模式直接切阻尼。

## 四、记录与学习检查

| 片段 | 首次观测 | 播报结果 | 播报后新观测 | 返回原因/任务状态 | 现场行为 | 退出 |
| --- | --- | --- | --- | --- | --- | --- |
| 导览 | | | | | | |
| 踢球 | | | | | | |

1. 播报成功而目标已离开时，后续程序应怎样处理？指出两种片段中的对应行。
2. 什么需求值得增加L3工作线程？仅在播报后检查目标时，顺序程序能否完成任务？
3. 致意返回CANCELLED、踢球窗口达到上限，分别应怎样记录？为什么都不能直接填“表演成功”？
4. 若修改为一段完整应用，仍需确定哪些对象、场地、模块交接和验收要求？列出本组实际需要解决的项目问题。

**完成核对**：至少完成可用片段的交接记录，另一类按设备安排验证；未运行项明确标记。没有留下活动管理器、音频或动作任务。